# Beijing AQI: Clean vs. Polluted Next-Day Model

This notebook trains a binary XGBoost classifier to predict whether the next calendar day will be **Clean** or **Polluted**, using completed same-day pollutant and ERA5 weather summaries from MySQL table `air_quality.aqi_daily`.

**Target rule:** Clean means next-day AQI ≤ 100; Polluted means next-day AQI > 100. The source table is never edited. Rows without a valid next-day AQI are omitted from the model examples only.


## Evaluation approach

The first 80% of unique dates are used for training and the final 20% are held out chronologically. Four expanding-window folds, grouped by date, choose XGBoost parameters using macro F1. Imputation is inside the pipeline. Results include a majority-class baseline, a persistence baseline (tomorrow is like today), a date-block bootstrap accuracy interval, per-class metrics, and a confusion matrix.

Features are today's six pollutants and nine daily weather summaries. No next-day weather or target-day information is used as a predictor.


In [ ]:
import json
import os
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sqlalchemy import create_engine, text
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder

try:
    import xgboost
    from xgboost import XGBClassifier
except ImportError as error:
    raise ImportError("Install XGBoost with `%pip install xgboost`, then restart the kernel.") from error

HERE = Path.cwd().resolve()
ROOT = next((p for p in [HERE, *HERE.parents]
             if (p / "air-quality-classification" / "app" / "aqi_labels.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or its `2 class` folder.")
APP_DIR = ROOT / "air-quality-classification" / "app"
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))
from aqi_labels import O3_1H_BREAKPOINTS, POLLUTANT_BREAKPOINTS, category_codes, interpolate_iaqi

OUT = ROOT / "2 class"
DB_URL = os.environ.get("DB_URL", "mysql+pymysql://root:@localhost/air_quality")
SOURCE_TABLE = "aqi_daily"
POLLUTANTS = ["PM2.5", "PM10", "SO2", "NO2", "CO", "O3"]
WEATHER = ["TEMP_mean", "TEMP_range", "PRES_mean", "DEWP_mean", "RH_mean",
           "RAIN_sum", "WSPM_mean", "wd_sin", "wd_cos"]
FEATURES = POLLUTANTS + WEATHER
CLASSES = ["Clean", "Polluted"]
RANDOM_STATE = 42


## 1. Load the daily database table

Read `aqi_daily` directly from MySQL. This notebook has no CSV fallback. It checks required fields and the unique station-day key before building labels.


In [ ]:
engine = create_engine(DB_URL, pool_pre_ping=True)
try:
    with engine.connect() as connection:
        source_df = pd.read_sql_query(text(f"SELECT * FROM {SOURCE_TABLE}"), connection)
finally:
    engine.dispose()

required = ["date", "station", "station_id", *POLLUTANTS, "O3_1h_max", *WEATHER]
missing = [column for column in required if column not in source_df.columns]
if missing:
    raise KeyError(f"air_quality.{SOURCE_TABLE} is missing columns: {missing}")
if source_df.duplicated(["station_id", "date"]).any():
    raise ValueError("aqi_daily contains duplicate station-days.")
source_df["date"] = pd.to_datetime(source_df["date"], errors="coerce").dt.normalize()
print(f"Loaded rows: {len(source_df):,}")
print(f"Stations: {source_df.station.nunique()}; dates: {source_df.date.min().date()} to {source_df.date.max().date()}")
display(source_df.head())


## 2. Calculate AQI labels and make exact next-day examples

Calculate pollutant sub-indices with the shared China AQI breakpoint tables. For O3 above 800 µg/m³, use the daily 1-hour maximum. CO in `aqi_daily` is in mg/m³. Use today's completed predictors to target the same station's next calendar-day AQI.


In [ ]:
daily = source_df.copy()
iaqi_columns = []
for pollutant in POLLUTANTS:
    values = pd.to_numeric(daily[pollutant], errors="coerce")
    column = f"{pollutant}_iaqi"
    if pollutant == "O3":
        subindex = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS["O3"])
        one_hour = interpolate_iaqi(pd.to_numeric(daily["O3_1h_max"], errors="coerce"), O3_1H_BREAKPOINTS)
        high_ozone = values.gt(800)
        subindex.loc[high_ozone] = one_hour.loc[high_ozone]
    else:
        subindex = interpolate_iaqi(values, POLLUTANT_BREAKPOINTS[pollutant])
    daily[column] = subindex.round()
    iaqi_columns.append(column)

daily["AQI"] = daily[iaqi_columns].max(axis=1, skipna=True)
daily["aqi_category_code"] = category_codes(daily["AQI"])

today = daily[["station_id", "station", "date", *FEATURES, "AQI"]].copy()
tomorrow = daily[["station_id", "date", "AQI"]].copy()
tomorrow["date"] = tomorrow["date"] - pd.Timedelta(days=1)
tomorrow = tomorrow.rename(columns={"AQI": "target_AQI"})
data = today.merge(tomorrow, on=["station_id", "date"], how="inner", validate="one_to_one")
data = data.dropna(subset=["target_AQI"]).copy()
data["target"] = np.where(data["target_AQI"] <= 100, "Clean", "Polluted")
data["today_target"] = np.where(data["AQI"] <= 100, "Clean", "Polluted")
data.loc[data["AQI"].isna(), "today_target"] = np.nan
data = data.sort_values(["date", "station"]).reset_index(drop=True)
print(f"Valid next-day examples: {len(data):,}")
print("Target class counts:")
display(data["target"].value_counts().reindex(CLASSES, fill_value=0).rename("rows").to_frame())
display(data[["date", "station", *FEATURES, "target_AQI", "target"]].head())


## 3. Chronological split and tune XGBoost

Keep all rows from each date together. Hyperparameter selection uses only the training dates; the final 20% of unique dates remain untouched until evaluation.


In [ ]:
dates = np.array(sorted(data.date.unique()))
cut = int(len(dates) * 0.80)
train_dates, test_dates = dates[:cut], dates[cut:]
train = data[data.date.isin(train_dates)].copy()
test = data[data.date.isin(test_dates)].copy()
label_encoder = LabelEncoder().fit(train.target)
y_train = label_encoder.transform(train.target)
y_test = label_encoder.transform(test.target)
X_train, X_test = train[FEATURES], test[FEATURES]
print(f"Train: {len(train):,} rows through {train.date.max().date()}")
print(f"Test: {len(test):,} rows, {test.date.min().date()} through {test.date.max().date()}")

train_date_codes = pd.factorize(train.date, sort=True)[0]
folds = []
for left_dates, right_dates in TimeSeriesSplit(n_splits=4).split(np.unique(train_date_codes)):
    folds.append((np.flatnonzero(np.isin(train_date_codes, left_dates)),
                  np.flatnonzero(np.isin(train_date_codes, right_dates))))

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("classifier", XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                                  random_state=RANDOM_STATE, n_jobs=4,
                                  tree_method="hist", verbosity=0)),
])
parameter_grid = {
    "classifier__n_estimators": [120, 240],
    "classifier__max_depth": [3, 5],
    "classifier__learning_rate": [0.06],
}
search = GridSearchCV(pipeline, parameter_grid, scoring="f1_macro", cv=folds,
                      n_jobs=1, refit=True, verbose=1)
search.fit(X_train, y_train)
model = search.best_estimator_
print("Best parameters:", search.best_params_)
print(f"Best training CV macro F1: {search.best_score_:.4f}")


## 4. Held-out evaluation

Compare XGBoost with majority and persistence baselines on exactly the same test rows. Persistence uses today's binary AQI class; if today's AQI is unavailable, it falls back to the training majority class. The confidence interval resamples whole test dates to account for the shared date conditions across stations.


In [ ]:
predicted_codes = model.predict(X_test).astype(int)
predicted = label_encoder.inverse_transform(predicted_codes)
truth = test.target.to_numpy()
majority = train.target.value_counts().idxmax()
majority_prediction = np.repeat(majority, len(test))
persistence = test.today_target.fillna(majority).to_numpy()

unique_test_dates = np.unique(test.date.to_numpy())
indices_by_date = {date: np.flatnonzero(test.date.to_numpy() == date) for date in unique_test_dates}
rng = np.random.default_rng(RANDOM_STATE)
bootstrap_accuracy = []
for _ in range(1000):
    sampled_dates = rng.choice(unique_test_dates, size=len(unique_test_dates), replace=True)
    indices = np.concatenate([indices_by_date[date] for date in sampled_dates])
    bootstrap_accuracy.append(accuracy_score(truth[indices], predicted[indices]))
ci_low, ci_high = np.quantile(bootstrap_accuracy, [0.025, 0.975])

metrics = {
    "accuracy": accuracy_score(truth, predicted),
    "accuracy_ci95_low": ci_low,
    "accuracy_ci95_high": ci_high,
    "balanced_accuracy": balanced_accuracy_score(truth, predicted),
    "precision_macro": precision_score(truth, predicted, average="macro", zero_division=0),
    "recall_macro": recall_score(truth, predicted, average="macro", zero_division=0),
    "macro_f1": f1_score(truth, predicted, average="macro", zero_division=0),
    "majority_baseline_accuracy": accuracy_score(truth, majority_prediction),
    "persistence_accuracy": accuracy_score(truth, persistence),
    "test_rows": len(test),
}
display(pd.Series(metrics, name="value").round(4).to_frame())
print(classification_report(truth, predicted, labels=CLASSES, target_names=CLASSES,
                            digits=4, zero_division=0))
confusion = pd.DataFrame(confusion_matrix(truth, predicted, labels=CLASSES),
                         index=[f"Actual {label}" for label in CLASSES],
                         columns=[f"Predicted {label}" for label in CLASSES])
display(confusion)


## 5. Save model artifact and evaluation files

Save the trained XGBoost pipeline, its label encoder, feature order, thresholds, date ranges, and measured metrics in this folder. The Streamlit app uses this artifact and requires the notebook to be run successfully first.


In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
results = pd.DataFrame([{"model": "XGBoost", **metrics,
                         "training_rows": len(train),
                         "train_start": str(train.date.min().date()),
                         "train_end": str(train.date.max().date()),
                         "test_start": str(test.date.min().date()),
                         "test_end": str(test.date.max().date()),
                         "best_parameters": json.dumps(search.best_params_)}])
class_report = pd.DataFrame(classification_report(
    truth, predicted, labels=CLASSES, target_names=CLASSES,
    output_dict=True, zero_division=0)).T
confusion_long = confusion.rename_axis("actual").reset_index().melt(
    id_vars="actual", var_name="predicted", value_name="count")
per_station = pd.DataFrame([
    {"station": station, "rows": len(group),
     "accuracy": accuracy_score(group.target, predicted[group.index.to_numpy() - test.index.min()])}
    for station, group in test.groupby("station")
])
results.to_csv(OUT / "results.csv", index=False)
class_report.to_csv(OUT / "per_class.csv")
confusion_long.to_csv(OUT / "confusion_matrix.csv", index=False)
per_station.to_csv(OUT / "per_station.csv", index=False)
artifact = {
    "model": model,
    "label_encoder": label_encoder,
    "classifier_name": "XGBClassifier",
    "features": FEATURES,
    "classes": CLASSES,
    "clean_threshold_aqi": 100,
    "polluted_threshold_aqi_exclusive": 100,
    "target_horizon": "next calendar day",
    "evaluation_metrics": {key: float(value) for key, value in metrics.items()},
    "best_parameters": search.best_params_,
    "training_period": [str(train.date.min().date()), str(train.date.max().date())],
    "test_period": [str(test.date.min().date()), str(test.date.max().date())],
    "xgboost_version": xgboost.__version__,
}
joblib.dump(artifact, OUT / "2class_model.joblib")
print(f"Saved model and evaluation files under {OUT}")


## Interpretation and limitations

Accuracy is the overall fraction of test examples classified correctly. Also inspect polluted-class recall: false Clean predictions are particularly important for a pollution warning feature. The 70% target is not, by itself, enough to establish operational reliability; compare with both baselines and examine the confidence interval and class metrics.

This model uses same-day observed weather, which is only available after that day's measurements are complete. It does not use tomorrow's weather forecast. The model is a historical Beijing benchmark, not an official AQI report or health advisory.
